# Call the Covertype prediction API

Read one real held-out example from the test split and send its measured values to the API. The API service must be running and a trained model must have been uploaded to MinIO.

In [3]:
import os
import pandas as pd
import psycopg
import requests
from psycopg.rows import dict_row

DB_CONFIG = {
    "host": os.getenv("PROJECT_DATA_DB_HOST", "localhost"),
    "port": int(os.getenv("PROJECT_DATA_DB_PORT", "5433")),
    "dbname": os.getenv("PROJECT_DATA_DB_NAME", "cover_data"),
    "user": os.getenv("PROJECT_DATA_DB_USER", "cover_user"),
    "password": os.getenv("PROJECT_DATA_DB_PASSWORD", "cover_password"),
}
API_BASE_URL = os.getenv("API_BASE_URL", "http://localhost:8000").rstrip("/")
FEATURES = [
    "elevation", "aspect", "slope", "horizontal_distance_to_hydrology",
    "vertical_distance_to_hydrology", "horizontal_distance_to_roadways",
    "hillshade_9am", "hillshade_noon", "hillshade_3pm",
    "horizontal_distance_to_fire_points", "wilderness_area", "soil_type",
]
with psycopg.connect(**DB_CONFIG, row_factory=dict_row) as conn:
    sample = pd.DataFrame(conn.execute(
        f"SELECT {', '.join(FEATURES)}, cover_type FROM cover_prepared WHERE split = 'test' ORDER BY prepared_id LIMIT 1"
    ).fetchall())
if sample.empty:
    raise RuntimeError("No test example found. Run 01_prepare.ipynb first.")


## Check service and predict

In [4]:
health = requests.get(f"{API_BASE_URL}/health", timeout=10)
health.raise_for_status()
print("API health:", health.json())
example = sample.iloc[0]
payload = {"features": {name: example[name].item() if hasattr(example[name], "item") else example[name] for name in FEATURES}}
response = requests.post(f"{API_BASE_URL}/predict", json=payload, timeout=60)
response.raise_for_status()
result = response.json()
print("Input feature values:")
display(pd.DataFrame([payload["features"]]))
print("Actual cover type:", int(example["cover_type"]))
print("API prediction:", result)


API health: {'status': 'ok', 'model_loaded': True}
Input feature values:


,elevation,aspect,slope,horizontal_distance_to_hydrology,vertical_distance_to_hydrology,horizontal_distance_to_roadways,hillshade_9am,hillshade_noon,hillshade_3pm,horizontal_distance_to_fire_points,wilderness_area,soil_type
0,2941,52,9,30,-3,1566,225,221,130,3399,Rawah,C7202


Actual cover type: 1
API prediction: {'predicted_cover_type': 1}
